# Model Load and Verification

This notebook loads the exported models from disk and verifies that they run inference correctly on a few samples.


**Section: Setup**

In [ ]:
import json
from pathlib import Path
import numpy as np
import tensorflow as tf

DATA_DIR = Path('dataset')
MODELS_DIR = Path('models')
IMG_SIZE = 224


**Section: Load Class Names**

In [ ]:
class_names_path = MODELS_DIR / 'class_names.json'
if not class_names_path.exists():
    raise FileNotFoundError(f"Missing {class_names_path}")

class_names = json.loads(class_names_path.read_text())
print('Classes:', class_names)


**Section: Sample Inputs**

In [ ]:
# Collect a few sample images for testing
image_paths = []
for cls in class_names:
    cls_dir = DATA_DIR / cls
    if cls_dir.exists():
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
            image_paths.extend(cls_dir.glob(ext))

if not image_paths:
    raise FileNotFoundError('No images found in dataset/')

sample_paths = image_paths[:8]
print('Using sample images:', [str(p) for p in sample_paths])

def preprocess_for_model(path):
    img = tf.io.read_file(str(path))
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img

batch = tf.stack([preprocess_for_model(p) for p in sample_paths])


**Section: Load Keras (.keras)**

In [ ]:
keras_path = MODELS_DIR / 'pest_classifier.keras'
if keras_path.exists():
    keras_model = tf.keras.models.load_model(keras_path)
    preds = keras_model.predict(batch, verbose=0)
    print('Keras .keras loaded. Output shape:', preds.shape)
else:
    print('Keras .keras not found:', keras_path)


**Section: Load H5 (Full or Weights)**

In [ ]:
h5_path = MODELS_DIR / 'pest_classifier.h5'
weights_path = MODELS_DIR / 'pest_classifier.weights.h5'

h5_loaded = False
if h5_path.exists():
    try:
        h5_model = tf.keras.models.load_model(h5_path)
        preds = h5_model.predict(batch, verbose=0)
        print('H5 full model loaded. Output shape:', preds.shape)
        h5_loaded = True
    except Exception as e:
        print('Failed to load full H5 model:', e)

if not h5_loaded and weights_path.exists():
    # Rebuild the model architecture to load weights
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet'
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.layers.Rescaling(1./127.5, offset=-1)(inputs)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
    weights_model = tf.keras.Model(inputs, outputs)

    weights_model.load_weights(weights_path)
    preds = weights_model.predict(batch, verbose=0)
    print('Weights-only model loaded. Output shape:', preds.shape)


**Section: Load SavedModel**

In [ ]:
saved_model_dir = MODELS_DIR / 'pest_classifier_savedmodel'
if saved_model_dir.exists():
    sm = tf.saved_model.load(str(saved_model_dir))
    infer = sm.signatures['serving_default']
    outputs = infer(tf.constant(batch))
    # Take first output tensor
    out = list(outputs.values())[0].numpy()
    print('SavedModel loaded. Output shape:', out.shape)
else:
    print('SavedModel not found:', saved_model_dir)


**Section: Load TFLite**

In [ ]:
tflite_path = MODELS_DIR / 'pest_classifier.tflite'
if tflite_path.exists():
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # TFLite expects a fixed input shape (often batch size 1)
    input_shape = input_details[0]['shape']
    batch_size = int(input_shape[0])

    # Use a single sample if model expects batch size 1
    input_data = batch.numpy()
    if batch_size == 1:
        input_data = input_data[:1]
    else:
        input_data = input_data[:batch_size]

    input_data = input_data.astype(input_details[0]['dtype'])
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    print('TFLite model loaded. Output shape:', output_data.shape)
else:
    print('TFLite not found:', tflite_path)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

best_model_path = MODELS_DIR / 'best_model.keras'
if not best_model_path.exists():
    raise FileNotFoundError(f"Missing {best_model_path}")

best_model = tf.keras.models.load_model(best_model_path)
needs_preprocess = not any(
    isinstance(layer, (tf.keras.layers.Rescaling, tf.keras.layers.Normalization))
    for layer in best_model.layers
)
print('best_model loaded. Needs MobileNetV2 preprocess:', needs_preprocess)

# Build dataset splits
all_paths = []
all_labels = []
for idx, cls in enumerate(class_names):
    cls_dir = DATA_DIR / cls
    if not cls_dir.exists():
        continue
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        for p in cls_dir.glob(ext):
            all_paths.append(str(p))
            all_labels.append(idx)

if not all_paths:
    raise FileNotFoundError('No images found in dataset/')

all_paths = np.array(all_paths)
all_labels = np.array(all_labels)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, stratify=all_labels, random_state=42
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=42
)

NUM_CLASSES = len(class_names)

def load_eval_img(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    if needs_preprocess:
        img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

BATCH_SIZE = 32

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
    .map(load_eval_img, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
    .map(load_eval_img, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print('Validation samples:', len(val_paths))
print('Test samples:', len(test_paths))

def print_eval_results(prefix, results, names):
    if not isinstance(results, (list, tuple)):
        results = [results]
    print(f"{prefix} metrics:")
    for name, value in zip(names, results):
        try:
            print(f"  {name}: {value:.4f}")
        except Exception:
            print(f"  {name}: {value}")

val_results = best_model.evaluate(val_ds, verbose=0)
test_results = best_model.evaluate(test_ds, verbose=0)
metric_names = best_model.metrics_names

print_eval_results('Validation', val_results, metric_names)
print_eval_results('Test', test_results, metric_names)

# Detailed classification report on test set
y_true = []
y_pred = []
for batch_x, batch_y in test_ds:
    preds = best_model.predict(batch_x, verbose=0)
    y_true.extend(np.argmax(batch_y.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print('Classification report (test set):')
print(classification_report(y_true, y_pred, target_names=class_names))
print('Confusion matrix:')
print(confusion_matrix(y_true, y_pred))
